# CareFlow — AI Care Coordination Assistant

**IIT Hyderabad Applied AI Professional Certification · Capstone**

This notebook is one milestone in the eight-milestone CareFlow implementation.
All patient, insurance, appointment and referral examples are synthetic. No real PHI is used.

> **Data boundary:** the official CareFlow Data Toolkit assets are the source of truth.
> Intake records feed M1; mock API tables feed M2/M6; the policy corpus feeds M3/M4;
> the golden evaluation set is finalized in M8 and should be started at M4.

## M5 — Stateful LangGraph Care Coordination Workflow

### Goal
Connect the M1 intake boundary, M2 deterministic read tools, and M4 policy retrieval in a real, typed LangGraph workflow. This notebook is a deterministic educational baseline: it does not call an LLM, diagnose, provide treatment advice, or perform writes.

The graph uses a state field to make routing decisions, pure conditional-edge routers, a resumable clarification interrupt for missing patient identity, and an in-memory checkpointer for the demo. The synthetic patient reference supplied on resume is **not** authentication; production must enforce identity and authorization outside the graph.

Optional local settings are loaded from the project-root `.env` file. To render the workflow diagram, install the Python `graphviz` package and the Graphviz system executable (`brew install graphviz` on macOS).

In [1]:
import csv
import json
import re
from pathlib import Path
from typing import Any, TypedDict

from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

DATA_RELATIVE_PATH = Path("data/careflow")
PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / DATA_RELATIVE_PATH).is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Cannot find {DATA_RELATIVE_PATH} from notebook working directory {Path.cwd()}"
    )

load_dotenv(PROJECT_ROOT / ".env", override=False)

DATA_DIR = PROJECT_ROOT / DATA_RELATIVE_PATH
CORPUS_DIR = DATA_DIR / "corpus" / "markdown"
documents = [
    {"source": path.name, "text": path.read_text(encoding="utf-8")}
    for path in sorted(CORPUS_DIR.glob("*.md"))
]
if not documents:
    raise FileNotFoundError(f"No Markdown policy documents found in {CORPUS_DIR.resolve()}")

def load_records(name: str) -> list[dict[str, Any]] | None:
    """Load one generated mock table from CSV or JSON, if present."""
    base = DATA_DIR / "mock_api"
    csv_path = base / f"{name}.csv"
    json_path = base / f"{name}.json"
    if csv_path.is_file():
        with csv_path.open(encoding="utf-8", newline="") as handle:
            return list(csv.DictReader(handle))
    if json_path.is_file():
        content = json.loads(json_path.read_text(encoding="utf-8"))
        if isinstance(content, list) and all(isinstance(row, dict) for row in content):
            return content
        raise ValueError(f"Expected a JSON list of records in {json_path}")
    return None

tables = {name: load_records(name) for name in ("plans", "patients", "appointments", "referrals")}
missing_tables = [name for name, rows in tables.items() if rows is None]
USING_DEMO_TABLES = bool(missing_tables)
if USING_DEMO_TABLES:
    print(f"Using synthetic demo rows; mock tables missing: {', '.join(missing_tables)}")
    tables = {
        "plans": [{"plan_code": "MERIDIAN-GOLD", "annual_deductible_usd": 500, "coinsurance_rate": 0.20,
                   "specialist_copay_usd": 25, "telehealth_copay_usd": 10}],
        "patients": [{"patient_ref": "MHP-P-00001", "plan_code": "MERIDIAN-GOLD",
                      "eligibility_status": "active", "deductible_met_usd": 250}],
        "appointments": [{"appointment_id": "MHP-A-000001", "patient_ref": "MHP-P-00001",
                           "specialty": "Cardiology", "scheduled_for": "2026-10-15T10:00",
                           "status": "scheduled"}],
        "referrals": [{"referral_id": "MHP-R-00001", "patient_ref": "MHP-P-00001",
                        "to_specialty": "Cardiology", "urgency": "Routine", "status": "open"}],
    }

print(f"Policy documents loaded: {len(documents)} from {CORPUS_DIR.resolve()}")
print("Mock API mode:", "demo fixtures" if USING_DEMO_TABLES else "generated data")

Policy documents loaded: 14 from /Users/sgunaseelan/Projects/API/AI-IIT/applied-ai-capstone-starter/data/careflow/corpus/markdown
Mock API mode: generated data


In [4]:
class CareFlowState(TypedDict, total=False):
    raw_message: str
    patient_ref: str
    intent: str
    route: str
    retrieved: list[dict[str, Any]]
    tool_result: dict[str, Any]
    answer: str
    human_review_required: bool
    events: list[str]

STOP_WORDS = {"the", "and", "for", "with", "that", "this", "from", "are", "was", "were", "what", "how", "which"}
CLINICAL_MARKERS = (
    "should i", "dosage", "dose", "medication", "take another tablet", "symptom",
    "rash", "diagnose", "diagnosis", "treatment", "prescription", "what is wrong with me",
)

def terms(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9-]+", text.lower())) - STOP_WORDS

policy_chunks: list[dict[str, str]] = []
for document in documents:
    words = document["text"].split()
    start = 0
    while start < len(words):
        end = min(len(words), start + 180)
        policy_chunks.append({
            "source": document["source"],
            "text": " ".join(words[start:end]),
            "chunk_id": f"{document['source']}:{start // 145:03d}",
        })
        if end == len(words):
            break
        start = end - 35

def retrieve_policy(question: str, k: int = 4) -> list[dict[str, str]]:
    query_terms = terms(question)
    ranked = []
    for chunk in policy_chunks:
        score = len(query_terms & terms(chunk["text"]))
        if score:
            ranked.append((score, chunk))
    ranked.sort(key=lambda item: item[0], reverse=True)
    return [chunk for _, chunk in ranked[:k]]

def route_after_intake(state: CareFlowState) -> str:
    """Pure router: the parse node writes intent; this edge only reads it."""
    if state["intent"] == "clinical":
        return "clinical_refusal"
    if state["intent"] == "policy":
        return "retrieve_policy"
    return "tool_lookup"

def parse_intake(state: CareFlowState) -> dict[str, Any]:
    message = state["raw_message"].strip()
    if not message:
        raise ValueError("raw_message must not be empty")
    lowered = message.lower()
    patient_match = re.search(r"\bMHP-P-\d+\b", message, flags=re.IGNORECASE)
    patient_ref = patient_match.group(0).upper() if patient_match else state.get("patient_ref", "")
    plan_match = re.search(r"\b(?:MERIDIAN-[A-Z]+|CIVIC-BASE)\b", message, flags=re.IGNORECASE)

    if any(marker in lowered for marker in CLINICAL_MARKERS):
        intent = "clinical"
    elif plan_match and any(word in lowered for word in ("plan", "benefit", "copay", "co-pay", "deductible", "coverage")):
        intent = "plan"
    elif any(word in lowered for word in ("policy", "copay", "co-pay", "coverage", "benefit", "coinsurance", "deductible", "preauth", "pre-auth", "how long")):
        intent = "policy"
    elif any(word in lowered for word in ("appointment", "referral", "eligibility", "patient record")):
        intent = "appointments" if "appointment" in lowered else "referrals" if "referral" in lowered else "patient"
    else:
        intent = "policy"

    return {
        "patient_ref": patient_ref,
        "plan_code": plan_match.group(0).upper() if plan_match else "",
        "intent": intent,
        "route": intent,
        "events": state.get("events", []) + ["parse_intake"],
    }

def clinical_refusal(state: CareFlowState) -> dict[str, Any]:
    return {
        "route": "refuse_and_human_review",
        "human_review_required": True,
        "answer": (
            "I cannot interpret symptoms or provide diagnosis, treatment, or medication advice. "
            "Please contact a licensed clinician; this request requires human review."
        ),
        "events": state.get("events", []) + ["clinical_refusal"],
    }

def retrieve_policy_node(state: CareFlowState) -> dict[str, Any]:
    hits = retrieve_policy(state["raw_message"])
    return {
        "retrieved": hits,
        "events": state.get("events", []) + ["retrieve_policy"],
    }

def lookup_records(table: str, field: str, value: str) -> list[dict[str, Any]]:
    return [row for row in tables[table] if str(row.get(field, "")).casefold() == value.casefold()]

def tool_lookup(state: CareFlowState) -> dict[str, Any]:
    intent = state["intent"]
    patient_ref = state.get("patient_ref", "")
    if intent == "plan":
        records = lookup_records("plans", "plan_code", state.get("plan_code", ""))
        tool_name = "get_plan"
    else:
        if not patient_ref:
            clarification = interrupt({
                "kind": "clarification",
                "question": "Which synthetic patient reference should I use?",
                "required_field": "patient_ref",
                "example": "MHP-P-00001",
                "reason": f"A patient reference is required for the {intent} lookup.",
            })
            if not isinstance(clarification, dict):
                raise ValueError("Resume value must be an object containing patient_ref")
            patient_ref = str(clarification.get("patient_ref", "")).upper()
            if not re.fullmatch(r"MHP-P-\d+", patient_ref):
                raise ValueError("Resume value must contain a valid synthetic patient_ref (MHP-P-<digits>)")

        patient = lookup_records("patients", "patient_ref", patient_ref)
        if not patient:
            records = []
            tool_name = f"get_{intent}"
        elif intent == "appointments":
            records = lookup_records("appointments", "patient_ref", patient_ref)
            tool_name = "get_appointments"
        elif intent == "referrals":
            records = lookup_records("referrals", "patient_ref", patient_ref)
            tool_name = "get_referrals"
        else:
            records = patient
            tool_name = "get_patient"

    return {
        "patient_ref": patient_ref,
        "tool_result": {"tool_name": tool_name, "records": records},
        "events": state.get("events", []) + [tool_name],
    }

def route_after_retrieval(state: CareFlowState) -> str:
    return "respond_policy" if state.get("retrieved") else "policy_not_found"

def respond_policy(state: CareFlowState) -> dict[str, Any]:
    citations = "\n\n".join(
        f"[{index}] {hit['source']} ({hit['chunk_id']})\n{hit['text'][:700]}"
        for index, hit in enumerate(state["retrieved"], start=1)
    )
    return {
        "answer": "Retrieved policy excerpts (verify applicability; this baseline does not generate a free-form answer):\n\n" + citations,
        "route": "policy_context_found",
        "events": state.get("events", []) + ["respond_policy"],
    }

def policy_not_found(state: CareFlowState) -> dict[str, Any]:
    return {
        "route": "human_review",
        "human_review_required": True,
        "answer": "I could not find supporting policy in the loaded corpus, so I will not guess. Please route this question to clinic operations.",
        "events": state.get("events", []) + ["policy_not_found"],
    }

def respond_tool(state: CareFlowState) -> dict[str, Any]:
    result = state["tool_result"]
    records = result["records"]
    if not records:
        answer = f"No matching record was found for {result['tool_name']}."
    elif result["tool_name"] == "get_plan":
        plan = records[0]
        answer = (
            f"{plan.get('plan_code')}: specialist co-pay ${plan.get('specialist_copay_usd', 'not listed')}; "
            f"annual deductible ${plan.get('annual_deductible_usd', 'not listed')}; "
            f"coinsurance {plan.get('coinsurance_rate', 'not listed')}."
        )
    else:
        answer = f"{result['tool_name']} returned {len(records)} matching record(s):\n" + json.dumps(records[:5], indent=2, default=str)
    return {
        "answer": answer,
        "route": "operational_tool_result",
        "events": state.get("events", []) + ["respond_tool"],
    }

builder = StateGraph(CareFlowState)
builder.add_node("parse_intake", parse_intake)
builder.add_node("clinical_refusal", clinical_refusal)
builder.add_node("retrieve_policy", retrieve_policy_node)
builder.add_node("policy_not_found", policy_not_found)
builder.add_node("respond_policy", respond_policy)
builder.add_node("tool_lookup", tool_lookup)
builder.add_node("respond_tool", respond_tool)

builder.add_edge(START, "parse_intake")
builder.add_conditional_edges(
    "parse_intake",
    route_after_intake,
    {"clinical_refusal": "clinical_refusal", "retrieve_policy": "retrieve_policy", "tool_lookup": "tool_lookup"},
)
builder.add_edge("clinical_refusal", END)
builder.add_conditional_edges(
    "retrieve_policy",
    route_after_retrieval,
    {"respond_policy": "respond_policy", "policy_not_found": "policy_not_found"},
)
builder.add_edge("respond_policy", END)
builder.add_edge("policy_not_found", END)
builder.add_edge("tool_lookup", "respond_tool")
builder.add_edge("respond_tool", END)

checkpointer = InMemorySaver()
careflow_graph = builder.compile(checkpointer=checkpointer)
print("Compiled CareFlow LangGraph with state-based routing and an in-memory checkpointer.")

# Clinical-advice requests are refused before retrieval or operational tools run.
clinical_result = careflow_graph.invoke({
    "raw_message": "Can you tell me if I should take another tablet?"
}, config={"configurable": {"thread_id": "careflow-clinical-demo"}})
assert clinical_result["route"] == "refuse_and_human_review"
assert clinical_result["events"] == ["parse_intake", "clinical_refusal"]
print("Clinical safety route:", clinical_result["route"], clinical_result["events"])

# Policy questions use M4-style lexical retrieval and retain source provenance.
policy_result = careflow_graph.invoke({
    "raw_message": "What does the policy say about routine referral processing?"
}, config={"configurable": {"thread_id": "careflow-policy-demo"}})
assert policy_result["route"] == "policy_context_found"
assert policy_result["retrieved"] and policy_result["retrieved"][0]["source"]
print("Policy route:", policy_result["route"], "sources:", sorted({hit["source"] for hit in policy_result["retrieved"]}))

# Missing patient identity pauses at a resumable clarification interrupt.
thread_config = {"configurable": {"thread_id": "careflow-m5-appointment-demo"}}
paused = careflow_graph.invoke({
    "raw_message": "Show my next Cardiology appointment"
}, config=thread_config)
interrupt_payload = paused["__interrupt__"][0].value
print("Clarification interrupt:", interrupt_payload)
appointment_result = careflow_graph.invoke(
    Command(resume={"patient_ref": "MHP-P-00001"}),
    config=thread_config,
)
assert appointment_result["route"] == "operational_tool_result"
assert appointment_result["tool_result"]["tool_name"] == "get_appointments"
print("Resumed tool route:", appointment_result["route"], appointment_result["events"])
print(appointment_result["answer"])

Compiled CareFlow LangGraph with state-based routing and an in-memory checkpointer.


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse_intake(parse_intake)
	clinical_refusal(clinical_refusal)
	retrieve_policy(retrieve_policy)
	policy_not_found(policy_not_found)
	respond_policy(respond_policy)
	tool_lookup(tool_lookup)
	respond_tool(respond_tool)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse_intake;
	parse_intake -.-> clinical_refusal;
	parse_intake -.-> retrieve_policy;
	parse_intake -.-> tool_lookup;
	retrieve_policy -.-> policy_not_found;
	retrieve_policy -.-> respond_policy;
	tool_lookup --> respond_tool;
	clinical_refusal --> __end__;
	policy_not_found --> __end__;
	respond_policy --> __end__;
	respond_tool --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse_intake(parse_intake)
	clinical_refusal(clinical_refusal)
	retrieve_policy(retrieve_policy)
	policy_not_found(policy_not_found)
	respond_policy(respond_policy)
	tool_lookup(tool_lookup)
	respond_tool(respond_tool)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse_intake;
	parse_intake -.-> clinical_refusal;
	parse_intake -.-> retrieve_policy;
	parse_intake -.-> tool_lookup;
	retrieve_policy -.-> policy_not_found;
	retrieve_policy -.-> respond_policy;
	tool_lookup --> respond_tool;
	clinical_refusal --> __end__;
	policy_not_found --> __end__;
	respond_policy --> __end__;
	respond_tool --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

Clinical safety route: refuse_and_human_review ['parse_intake', 'clinical_refusal']
Policy route: policy_context_found sources: ['referral-policy.md', 'specialty-cardiology.md', 'spe

### Render the compiled workflow graph

This renders the actual compiled LangGraph topology as an SVG. It requires the Python `graphviz` package and the Graphviz `dot` executable on `PATH`.

In [ ]:
from graphviz import Digraph
from IPython.display import SVG, display
from shutil import which

compiled_topology = careflow_graph.get_graph()
workflow_diagram = Digraph("CareFlow_M5", format="svg")
workflow_diagram.attr(rankdir="TB", bgcolor="transparent", pad="0.25", nodesep="0.35", ranksep="0.55")
workflow_diagram.attr("node", shape="box", style="rounded,filled", fontname="Helvetica", fontsize="11", margin="0.15,0.08")
workflow_diagram.attr("edge", fontname="Helvetica", fontsize="9")

for node_id, node in compiled_topology.nodes.items():
    label = {START: "START", END: "END"}.get(node_id, node.name or node_id)
    shape = "circle" if node_id in {START, END} else "box"
    workflow_diagram.node(node_id, label=label, shape=shape)

for edge in compiled_topology.edges:
    workflow_diagram.edge(
        edge.source,
        edge.target,
        label=str(edge.data) if edge.data is not None else "",
    )

if which("dot") is None:
    print("Graphviz rendering is unavailable: install the system Graphviz executable (macOS: brew install graphviz), then restart the notebook kernel.")
else:
    display(SVG(workflow_diagram.pipe(format="svg").decode("utf-8")))

### Why state matters

The state object makes routing decisions explicit and inspectable:

```text
INTAKE → ROUTE → TOOLS/RETRIEVAL → RESPONSE
```

A real LangGraph implementation can replace these functions with graph nodes and conditional edges without changing the state contract.